# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
from collections.abc import Iterable

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print out basic dataset metadata
print(f"Dataset: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their field @ids & names
print("Available record sets in the dataset:")

record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record set definitions found in metadata. Attempting to fetch from dataset schema...")
    # For exploration, we can try to load all defined record sets via dataset object
    # mlcroissant exposes them as dataset._metadata.recordSets, but let's use the public API
    if hasattr(dataset, 'record_sets'):
        record_sets = dataset.record_sets
    else:
        record_sets = []

if not record_sets:
    # Try to infer them programmatically via dataset.records().
    print("Falling back: Try to enumerate record sets via dataset.records().\n")
    print("If you know the @id of a record set (e.g., from documentation), use that below.")
else:
    for rs in record_sets:
        # rs is a RecordSet object
        print(f"- @id: {getattr(rs, '@id', getattr(rs, 'id', None))} | Name: {getattr(rs, 'name', None)}")
        if hasattr(rs, 'field') and rs.field:
            print("  Fields:")
            fields = rs.field if isinstance(rs.field, list) else [rs.field]
            for f in fields:
                f_id = getattr(f, '@id', getattr(f, 'id', None)) if hasattr(f, '__dict__') else f.get('@id', f.get('id', None))
                f_name = getattr(f, 'name', None) if hasattr(f, 'name', None) else f.get('name', None)
                print(f"    - @id: {f_id} | Name: {f_name}")
print("\nSample first 1-2 records from a record set (replace '<record_set_id>' with a real @id):")
# Example code (user should provide a real @id):
# for x in dataset.records(record_set='<record_set_id>'):
#     print(x)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Common record set candidates: look for ones named like 'Protein', 'results', etc.
# With `mlcroissant`, you typically need to know the correct record set @id as referenced in the Croissant schema.
# If not obvious, consult the documentation or previous output.

# Here we will attempt to list all tabular record sets for this dataset

record_set_ids = []
try:
    # If previously loaded, record_sets is a list of RecordSet objects, otherwise empty
    if hasattr(dataset, 'record_sets'):
        record_sets = dataset.record_sets
    elif hasattr(metadata, 'recordSet'):
        record_sets = metadata.recordSet
    else:
        record_sets = []
except Exception:
    record_sets = []

# If no record sets found, prompt the user accordingly
if not record_sets:
    print("No explicit record sets found. If you know a valid record set @id, try using it below.")

# Otherwise, enumerate available record set @ids
for rs in record_sets:
    # In mlcroissant 0.3+, the RecordSet object has '@id', 'name', etc.
    rs_id = getattr(rs, '@id', getattr(rs, 'id', None))
    record_set_ids.append(rs_id)
    print(f"RecordSet: {rs_id} | {getattr(rs,'name',None)}")

# If none found, use a placeholder; else use the first available for demonstration
if not record_set_ids:
    main_record_set_id = '<insert_record_set_@id_here>'  # User must edit as appropriate
    print("Using a generic placeholder record set @id.")
else:
    main_record_set_id = record_set_ids[0]
    print(f"Selected primary record set: {main_record_set_id}")

# Load records into pandas DataFrame(s)
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns & sample of the first DataFrame
if main_record_set_id in dataframes:
    print(f"Columns in {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"Dataframe for {main_record_set_id} not loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

If you are unsure which columns to use, refer back to the Data Extraction step or inspect dataframe columns.

In [ ]:
# Select a numeric field for analysis -- replace this with a real column name as needed.
# For demonstration, we select the first numeric column in the main record set.
import numpy as np

numeric_field = None
df = dataframes.get(main_record_set_id)
if df is not None:
    # Find numeric columns
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Pick first
        print(f"Numeric field selected: {numeric_field}")
    else:
        # Try to parse columns as float if possible
        for col in df.columns:
            # Try parse as float?
            try:
                df[col+'_float'] = pd.to_numeric(df[col], errors='coerce')
                if df[col+'_float'].notnull().sum() > 0:
                    numeric_field = col+'_float'
                    print(f"Numeric field selected (converted): {numeric_field}")
                    break
            except Exception:
                continue
    if numeric_field is not None:
        # Define a demo threshold: 10 (can be changed)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the selected numeric column
        mean, std = filtered_df[numeric_field].mean(), filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try group by a categorical field (first non-numeric)
        non_numeric = [col for col in df.columns if df[col].dtype == 'object']
        group_field = non_numeric[0] if non_numeric else None
        if group_field is not None and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (show means):")
            display(grouped_df.head())
    else:
        print("No numeric fields detected to analyze.")
else:
    print("No dataframe available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field and a boxplot by group if possible
if df is not None and numeric_field is not None:
    fig, axs = plt.subplots(1, 2, figsize=(12,4))
    sns.histplot(df[numeric_field], kde=True, ax=axs[0])
    axs[0].set_title(f'Histogram of {numeric_field}')
    # Grouping
    if 'group_field' in locals() and group_field:
        sns.boxplot(data=df, x=group_field, y=numeric_field, ax=axs[1])
        axs[1].set_title(f'{numeric_field} by {group_field}')
        axs[1].tick_params(axis='x', rotation=45)
    else:
        axs[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Insufficient data for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded metadata and tabular data from the Croissant-formatted FAIR^2 Mass Spectrometry dataset.
- Demonstrated how to enumerate available record sets and fields using `mlcroissant`, referencing all by their `@id`.
- Performed basic filtering, normalization, and grouping on numeric columns.
- Displayed a basic histogram and boxplot of a chosen numeric variable.

_For full and reproducible use, review the markdown/code cells above and replace placeholder values (e.g., record set `@id`) with those found for your version of the dataset, as the Croissant schema may have multiple granular tabular areas or field options._